<a href="https://colab.research.google.com/github/PHAMVANTU467/TH_DEEP_LEARNING_BUOI04/blob/main/B%C3%80I_T%E1%BA%ACP_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Bài 1 Xây dựng CNN nhận dạng ảnh trên CIFAR10



## 1. Kiểm tra GPU

In [ ]:
import tensorflow as tf

class GPUManager:
    """Quản lý và kiểm tra trạng thái GPU."""
    def __init__(self):
        pass

    def check_gpu(self):
        print("Kiểm tra thiết bị GPU...")
        gpus = tf.config.list_physical_devices('GPU')
        if gpus:
            try:
                # Đặt cấu hình GPU để chỉ sử dụng bộ nhớ khi cần
                for gpu in gpus:
                    tf.config.experimental.set_memory_growth(gpu, True)
                logical_gpus = tf.config.experimental.list_logical_devices('GPU')
                print(f"Tìm thấy {len(gpus)} GPU vật lý, {len(logical_gpus)} GPU logic.")
                print("GPU đã sẵn sàng để huấn luyện.")
                return True
            except RuntimeError as e:
                print(e)
                return False
        else:
            print("Không tìm thấy thiết bị GPU. Đảm bảo bạn đang chạy trên Google Colab GPU và đã kích hoạt Runtime Type 'GPU'.")
            return False

# Khởi tạo và kiểm tra GPU
gpu_manager = GPUManager()
if not gpu_manager.check_gpu():
    raise SystemExit("Không có GPU hoặc cấu hình GPU không thành công. Vui lòng kiểm tra lại môi trường.")

## 2. Tải dữ liệu và tiền xử lý

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

class DatasetLoader:
    """Tải và tiền xử lý dữ liệu CIFAR-10."""
    def __init__(self):
        (self.x_train, self.y_train), (self.x_test, self.y_test) = cifar10.load_data()
        self.class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                            'dog', 'frog', 'horse', 'ship', 'truck']
        print(f"Kích thước dữ liệu huấn luyện: x_train={self.x_train.shape}, y_train={self.y_train.shape}")
        print(f"Kích thước dữ liệu kiểm tra: x_test={self.x_test.shape}, y_test={self.y_test.shape}")

    def display_sample_images(self, num_images=10):
        """Hiển thị một số ảnh mẫu từ tập huấn luyện."""
        plt.figure(figsize=(10, 2))
        for i in range(num_images):
            plt.subplot(1, num_images, i + 1)
            plt.imshow(self.x_train[i])
            plt.title(self.class_names[self.y_train[i][0]])
            plt.axis('off')
        plt.suptitle(f'{num_images} ảnh mẫu từ CIFAR-10', y=1.05)
        plt.show()

    def preprocess_data(self):
        """Chuẩn hóa dữ liệu ảnh và mã hóa One-Hot nhãn."""
        # Chuẩn hóa ảnh về khoảng [0, 1]
        self.x_train = self.x_train.astype('float32') / 255.0
        self.x_test = self.x_test.astype('float32') / 255.0

        # One-Hot Encoding nhãn
        self.y_train = to_categorical(self.y_train, num_classes=len(self.class_names))
        self.y_test = to_categorical(self.y_test, num_classes=len(self.class_names))

        print("Dữ liệu đã được chuẩn hóa và nhãn đã được mã hóa One-Hot.")
        print(f"Kích thước dữ liệu huấn luyện sau tiền xử lý: x_train={self.x_train.shape}, y_train={self.y_train.shape}")
        print(f"Kích thước dữ liệu kiểm tra sau tiền xử lý: x_test={self.x_test.shape}, y_test={self.y_test.shape}")

        return self.x_train, self.y_train, self.x_test, self.y_test, self.class_names

# Khởi tạo DatasetLoader, hiển thị ảnh mẫu và tiền xử lý dữ liệu
dataset_loader = DatasetLoader()
dataset_loader.display_sample_images(num_images=10)
x_train, y_train, x_test, y_test, class_names = dataset_loader.preprocess_data()

## 3. Data Augmentation

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

class DataAugmentation:
    """Thực hiện Data Augmentation cho dữ liệu huấn luyện."""
    def __init__(self, x_train):
        self.datagen = ImageDataGenerator(
            rotation_range=15,
            width_shift_range=0.1,
            height_shift_range=0.1,
            horizontal_flip=True,
            zoom_range=0.1
        )
        self.datagen.fit(x_train)
        print("Đã cấu hình Data Augmentation.")

    def get_generator(self):
        """Trả về generator cho Data Augmentation."""
        return self.datagen

# Khởi tạo DataAugmentation
data_augmentor = DataAugmentation(x_train)
datagen = data_augmentor.get_generator()

## 4. Xây dựng mô hình CNN

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Dropout, Dense, Flatten
from tensorflow.keras.optimizers import Adam

class CNNModel:
    """Xây dựng và biên dịch mô hình CNN."""
    def __init__(self, input_shape, num_classes):
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.model = self._build_model()

    def _build_model(self):
        model = Sequential([
            Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same', input_shape=self.input_shape),
            BatchNormalization(),
            Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same'),
            BatchNormalization(),
            MaxPooling2D((2, 2)),
            Dropout(0.2),

            Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same'),
            BatchNormalization(),
            Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same'),
            BatchNormalization(),
            MaxPooling2D((2, 2)),
            Dropout(0.3),

            Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same'),
            BatchNormalization(),
            Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same'),
            BatchNormalization(),
            MaxPooling2D((2, 2)),
            Dropout(0.4),

            Flatten(),
            Dense(128, activation='relu', kernel_initializer='he_uniform'),
            BatchNormalization(),
            Dropout(0.5),
            Dense(self.num_classes, activation='softmax')
        ])

        # Tối ưu hóa Adam với learning rate ban đầu 0.001
        optimizer = Adam(learning_rate=0.001)
        model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
        print("Mô hình CNN đã được xây dựng và biên dịch.")
        model.summary()
        return model

    def get_model(self):
        """Trả về mô hình CNN đã được xây dựng."""
        return self.model

# Xây dựng mô hình CNN
cnn_model_builder = CNNModel(input_shape=x_train.shape[1:], num_classes=len(class_names))
model = cnn_model_builder.get_model()

## 5. Huấn luyện mô hình

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt

class ModelTrainer:
    """Huấn luyện mô hình CNN với các callbacks."""
    def __init__(self, model, datagen, x_train, y_train, x_test, y_test, num_classes, model_name='cifar10_cnn_model.keras'):
        self.model = model
        self.datagen = datagen
        self.x_train = x_train
        self.y_train = y_train
        self.x_test = x_test
        self.y_test = y_test
        self.num_classes = num_classes
        self.model_name = model_name
        self.history = None
        self.callbacks = self._get_callbacks()

    def _get_callbacks(self):
        early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
        model_checkpoint = ModelCheckpoint(filepath=self.model_name, monitor='val_accuracy', save_best_only=True, verbose=1)
        reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.00001, verbose=1)
        return [early_stopping, model_checkpoint, reduce_lr]

    def train(self, epochs=100, batch_size=64):
        print("Bắt đầu huấn luyện mô hình...")
        self.history = self.model.fit(
            self.datagen.flow(self.x_train, self.y_train, batch_size=batch_size),
            epochs=epochs,
            validation_data=(self.x_test, self.y_test),
            callbacks=self.callbacks,
            verbose=1
        )
        print("Huấn luyện hoàn tất.")
        return self.history

    def plot_history(self):
        """Vẽ biểu đồ Accuracy và Loss của quá trình huấn luyện."""
        if self.history is None:
            print("Chưa có lịch sử huấn luyện để vẽ biểu đồ.")
            return

        # Vẽ biểu đồ Accuracy
        plt.figure(figsize=(12, 4))
        plt.subplot(1, 2, 1)
        plt.plot(self.history.history['accuracy'], label='Training Accuracy')
        plt.plot(self.history.history['val_accuracy'], label='Validation Accuracy')
        plt.title('Training and Validation Accuracy')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.grid(True)

        # Vẽ biểu đồ Loss
        plt.subplot(1, 2, 2)
        plt.plot(self.history.history['loss'], label='Training Loss')
        plt.plot(self.history.history['val_loss'], label='Validation Loss')
        plt.title('Training and Validation Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True)

        plt.tight_layout()
        plt.show()

# Khởi tạo ModelTrainer và huấn luyện mô hình
model_trainer = ModelTrainer(
    model=model,
    datagen=datagen,
    x_train=x_train,
    y_train=y_train,
    x_test=x_test,
    y_test=y_test,
    num_classes=len(class_names)
)
history = model_trainer.train(epochs=100, batch_size=64)
model_trainer.plot_history()

## 6. Đánh giá mô hình

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import numpy as np

class ModelEvaluator:
    """Đánh giá mô hình đã huấn luyện và hiển thị kết quả."""
    def __init__(self, model, x_test, y_test, class_names):
        self.model = model
        self.x_test = x_test
        self.y_test = y_test
        self.class_names = class_names

    def evaluate(self):
        print("Đang đánh giá mô hình trên tập kiểm tra...")
        loss, accuracy = self.model.evaluate(self.x_test, self.y_test, verbose=0)
        print(f"Độ chính xác trên tập kiểm tra: {accuracy:.4f}")
        print(f"Loss trên tập kiểm tra: {loss:.4f}")
        return loss, accuracy

    def display_classification_report(self):
        print("Hiển thị Báo cáo phân loại:")
        y_pred = np.argmax(self.model.predict(self.x_test, verbose=0), axis=1)
        y_true = np.argmax(self.y_test, axis=1)
        print(classification_report(y_true, y_pred, target_names=self.class_names))

    def display_confusion_matrix(self):
        print("Hiển thị Ma trận nhầm lẫn:")
        y_pred = np.argmax(self.model.predict(self.x_test, verbose=0), axis=1)
        y_true = np.argmax(self.y_test, axis=1)
        cm = confusion_matrix(y_true, y_pred)

        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='g', cmap='Blues', xticklabels=self.class_names, yticklabels=self.class_names)
        plt.xlabel('Dự đoán')
        plt.ylabel('Thực tế')
        plt.title('Ma trận nhầm lẫn')
        plt.show()

# Khởi tạo ModelEvaluator và đánh giá mô hình
model_evaluator = ModelEvaluator(model, x_test, y_test, class_names)
model_evaluator.evaluate()
model_evaluator.display_classification_report()
model_evaluator.display_confusion_matrix()

## 7. Dự đoán ảnh ngẫu nhiên

In [ ]:
import random
import matplotlib.pyplot as plt

class Predictor:
    """Thực hiện dự đoán trên các ảnh ngẫu nhiên."""
    def __init__(self, model, x_test, y_test, class_names):
        self.model = model
        self.x_test = x_test
        self.y_test = y_test
        self.class_names = class_names

    def predict_random_images(self, num_predictions=10):
        print(f"Dự đoán {num_predictions} ảnh ngẫu nhiên từ tập kiểm tra...")
        plt.figure(figsize=(15, 3))
        for i in range(num_predictions):
            idx = random.randint(0, len(self.x_test) - 1)
            img = self.x_test[idx]
            true_label = self.class_names[np.argmax(self.y_test[idx])]

            # Thêm chiều batch cho ảnh để đưa vào model
            img_batch = np.expand_dims(img, axis=0)
            prediction = self.model.predict(img_batch, verbose=0)
            predicted_label = self.class_names[np.argmax(prediction)]
            confidence = np.max(prediction) * 100

            plt.subplot(1, num_predictions, i + 1)
            plt.imshow(img)
            title_color = 'green' if true_label == predicted_label else 'red'
            plt.title(f"True: {true_label}\nPred: {predicted_label}\n({confidence:.1f}%) ", color=title_color, fontsize=8)
            plt.axis('off')
        plt.suptitle(f'Dự đoán {num_predictions} ảnh ngẫu nhiên', y=1.05)
        plt.show()

# Khởi tạo Predictor và thực hiện dự đoán
predictor = Predictor(model, x_test, y_test, class_names)
predictor.predict_random_images(num_predictions=10)

# Xây dựng CNN nhận dạng ảnh trên Fashion-MNIST



## 2. Tải dữ liệu và tiền xử lý (Fashion-MNIST)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical

class DatasetLoaderFashionMNIST:
    """Tải và tiền xử lý dữ liệu Fashion-MNIST."""
    def __init__(self):
        (self.x_train_fm, self.y_train_fm), (self.x_test_fm, self.y_test_fm) = fashion_mnist.load_data()
        self.class_names_fm = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
                               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
        print(f"Kích thước dữ liệu huấn luyện Fashion-MNIST: x_train={self.x_train_fm.shape}, y_train={self.y_train_fm.shape}")
        print(f"Kích thước dữ liệu kiểm tra Fashion-MNIST: x_test={self.x_test_fm.shape}, y_test={self.y_test_fm.shape}")

    def display_sample_images(self, num_images=10):
        """Hiển thị một số ảnh mẫu từ tập huấn luyện Fashion-MNIST."""
        plt.figure(figsize=(10, 2))
        for i in range(num_images):
            plt.subplot(1, num_images, i + 1)
            plt.imshow(self.x_train_fm[i], cmap=plt.cm.binary)
            plt.title(self.class_names_fm[self.y_train_fm[i]])
            plt.axis('off')
        plt.suptitle(f'{num_images} ảnh mẫu từ Fashion-MNIST', y=1.05)
        plt.show()

    def preprocess_data(self):
        """Chuẩn hóa dữ liệu ảnh và mã hóa One-Hot nhãn cho Fashion-MNIST."""
        # Fashion-MNIST là ảnh xám (28x28), cần thêm chiều kênh để tương thích với Conv2D
        self.x_train_fm = self.x_train_fm.reshape((self.x_train_fm.shape[0], 28, 28, 1))
        self.x_test_fm = self.x_test_fm.reshape((self.x_test_fm.shape[0], 28, 28, 1))

        # Chuẩn hóa ảnh về khoảng [0, 1]
        self.x_train_fm = self.x_train_fm.astype('float32') / 255.0
        self.x_test_fm = self.x_test_fm.astype('float32') / 255.0

        # One-Hot Encoding nhãn
        self.y_train_fm = to_categorical(self.y_train_fm, num_classes=len(self.class_names_fm))
        self.y_test_fm = to_categorical(self.y_test_fm, num_classes=len(self.class_names_fm))

        print("Dữ liệu Fashion-MNIST đã được chuẩn hóa và nhãn đã được mã hóa One-Hot.")
        print(f"Kích thước dữ liệu huấn luyện sau tiền xử lý: x_train_fm={self.x_train_fm.shape}, y_train_fm={self.y_train_fm.shape}")
        print(f"Kích thước dữ liệu kiểm tra sau tiền xử lý: x_test_fm={self.x_test_fm.shape}, y_test_fm={self.y_test_fm.shape}")

        return self.x_train_fm, self.y_train_fm, self.x_test_fm, self.y_test_fm, self.class_names_fm

# Khởi tạo DatasetLoaderFashionMNIST, hiển thị ảnh mẫu và tiền xử lý dữ liệu
dataset_loader_fm = DatasetLoaderFashionMNIST()
dataset_loader_fm.display_sample_images(num_images=10)
x_train_fm, y_train_fm, x_test_fm, y_test_fm, class_names_fm = dataset_loader_fm.preprocess_data()

## 3. Xây dựng mô hình CNN (Fashion-MNIST)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Dropout, Dense, Flatten
from tensorflow.keras.optimizers import Adam

class CNNModelFashionMNIST:
    """Xây dựng và biên dịch mô hình CNN cho Fashion-MNIST."""
    def __init__(self, input_shape, num_classes):
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.model_fm = self._build_model()

    def _build_model(self):
        model_fm = Sequential([
            Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same', input_shape=self.input_shape),
            BatchNormalization(),
            MaxPooling2D((2, 2)),
            Dropout(0.25),

            Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same'),
            BatchNormalization(),
            MaxPooling2D((2, 2)),
            Dropout(0.25),

            Flatten(),
            Dense(128, activation='relu', kernel_initializer='he_uniform'),
            BatchNormalization(),
            Dropout(0.5),
            Dense(self.num_classes, activation='softmax')
        ])

        optimizer = Adam(learning_rate=0.001)
        model_fm.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
        print("Mô hình CNN cho Fashion-MNIST đã được xây dựng và biên dịch.")
        model_fm.summary()
        return model_fm

    def get_model(self):
        """Trả về mô hình CNN đã được xây dựng cho Fashion-MNIST."""
        return self.model_fm

# Xây dựng mô hình CNN cho Fashion-MNIST
cnn_model_builder_fm = CNNModelFashionMNIST(input_shape=x_train_fm.shape[1:], num_classes=len(class_names_fm))
model_fm = cnn_model_builder_fm.get_model()

Mô hình CNN cho Fashion-MNIST đã được xây dựng và biên dịch.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 28, 28, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 14, 14, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       401,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 422,538 (1.61 MB)

 Trainable params: 422,090 (1.61 MB)

 Non-trainable params: 448 (1.75 KB)

## 4. Huấn luyện mô hình (Fashion-MNIST)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt

class ModelTrainerFashionMNIST:
    """Huấn luyện mô hình CNN cho Fashion-MNIST với các callbacks."""
    def __init__(self, model, x_train, y_train, x_test, y_test, model_name='fashion_mnist_cnn_model.keras'):
        self.model = model
        self.x_train = x_train
        self.y_train = y_train
        self.x_test = x_test
        self.y_test = y_test
        self.model_name = model_name
        self.history = None
        self.callbacks = self._get_callbacks()

    def _get_callbacks(self):
        early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
        model_checkpoint = ModelCheckpoint(filepath=self.model_name, monitor='val_accuracy', save_best_only=True, verbose=1)
        reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.00001, verbose=1)
        return [early_stopping, model_checkpoint, reduce_lr]

    def train(self, epochs=50, batch_size=64):
        print("Bắt đầu huấn luyện mô hình Fashion-MNIST...")
        self.history = self.model.fit(
            self.x_train, self.y_train,
            epochs=epochs,
            batch_size=batch_size,
            validation_data=(self.x_test, self.y_test),
            callbacks=self.callbacks,
            verbose=1
        )
        print("Huấn luyện Fashion-MNIST hoàn tất.")
        return self.history

    def plot_history(self):
        """Vẽ biểu đồ Accuracy và Loss của quá trình huấn luyện Fashion-MNIST."""
        if self.history is None:
            print("Chưa có lịch sử huấn luyện Fashion-MNIST để vẽ biểu đồ.")
            return

        plt.figure(figsize=(12, 4))
        plt.subplot(1, 2, 1)
        plt.plot(self.history.history['accuracy'], label='Training Accuracy')
        plt.plot(self.history.history['val_accuracy'], label='Validation Accuracy')
        plt.title('Training and Validation Accuracy (Fashion-MNIST)')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.grid(True)

        plt.subplot(1, 2, 2)
        plt.plot(self.history.history['loss'], label='Training Loss')
        plt.plot(self.history.history['val_loss'], label='Validation Loss')
        plt.title('Training and Validation Loss (Fashion-MNIST)')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True)

        plt.tight_layout()
        plt.show()

# Khởi tạo ModelTrainerFashionMNIST và huấn luyện mô hình
model_trainer_fm = ModelTrainerFashionMNIST(
    model=model_fm,
    x_train=x_train_fm,
    y_train=y_train_fm,
    x_test=x_test_fm,
    y_test=y_test_fm,
    model_name='fashion_mnist_cnn_model.keras'
)
history_fm = model_trainer_fm.train(epochs=50, batch_size=64)
model_trainer_fm.plot_history()

## 5. Đánh giá mô hình (Fashion-MNIST)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import numpy as np

class ModelEvaluatorFashionMNIST:
    """Đánh giá mô hình đã huấn luyện và hiển thị kết quả cho Fashion-MNIST."""
    def __init__(self, model, x_test, y_test, class_names):
        self.model = model
        self.x_test = x_test
        self.y_test = y_test
        self.class_names = class_names

    def evaluate(self):
        print("Đang đánh giá mô hình Fashion-MNIST trên tập kiểm tra...")
        loss, accuracy = self.model.evaluate(self.x_test, self.y_test, verbose=0)
        print(f"Độ chính xác trên tập kiểm tra Fashion-MNIST: {accuracy:.4f}")
        print(f"Loss trên tập kiểm tra Fashion-MNIST: {loss:.4f}")
        return loss, accuracy

    def display_classification_report(self):
        print("Hiển thị Báo cáo phân loại cho Fashion-MNIST:")
        y_pred = np.argmax(self.model.predict(self.x_test, verbose=0), axis=1)
        y_true = np.argmax(self.y_test, axis=1)
        print(classification_report(y_true, y_pred, target_names=self.class_names))

    def display_confusion_matrix(self):
        print("Hiển thị Ma trận nhầm lẫn cho Fashion-MNIST:")
        y_pred = np.argmax(self.model.predict(self.x_test, verbose=0), axis=1)
        y_true = np.argmax(self.y_test, axis=1)
        cm = confusion_matrix(y_true, y_pred)

        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='g', cmap='Blues', xticklabels=self.class_names, yticklabels=self.class_names)
        plt.xlabel('Dự đoán')
        plt.ylabel('Thực tế')
        plt.title('Ma trận nhầm lẫn (Fashion-MNIST)')
        plt.show()

# Khởi tạo ModelEvaluatorFashionMNIST và đánh giá mô hình
model_evaluator_fm = ModelEvaluatorFashionMNIST(model_fm, x_test_fm, y_test_fm, class_names_fm)
model_evaluator_fm.evaluate()
model_evaluator_fm.display_classification_report()
model_evaluator_fm.display_confusion_matrix()

## 6. Dự đoán ảnh ngẫu nhiên (Fashion-MNIST)

In [ ]:
import random
import matplotlib.pyplot as plt

class PredictorFashionMNIST:
    """Thực hiện dự đoán trên các ảnh ngẫu nhiên cho Fashion-MNIST."""
    def __init__(self, model, x_test, y_test, class_names):
        self.model = model
        self.x_test = x_test
        self.y_test = y_test
        self.class_names = class_names

    def predict_random_images(self, num_predictions=10):
        print(f"Dự đoán {num_predictions} ảnh ngẫu nhiên từ tập kiểm tra Fashion-MNIST...")
        plt.figure(figsize=(15, 3))
        for i in range(num_predictions):
            idx = random.randint(0, len(self.x_test) - 1)
            img = self.x_test[idx]
            true_label = self.class_names[np.argmax(self.y_test[idx])]

            img_batch = np.expand_dims(img, axis=0)
            prediction = self.model.predict(img_batch, verbose=0)
            predicted_label = self.class_names[np.argmax(prediction)]
            confidence = np.max(prediction) * 100

            plt.subplot(1, num_predictions, i + 1)
            plt.imshow(img.squeeze(), cmap=plt.cm.binary) # Fashion-MNIST là ảnh xám
            title_color = 'green' if true_label == predicted_label else 'red'
            plt.title(f"True: {true_label}\nPred: {predicted_label}\n({confidence:.1f}%) ", color=title_color, fontsize=8)
            plt.axis('off')
        plt.suptitle(f'Dự đoán {num_predictions} ảnh ngẫu nhiên (Fashion-MNIST)', y=1.05)
        plt.show()

# Khởi tạo PredictorFashionMNIST và thực hiện dự đoán
predictor_fm = PredictorFashionMNIST(model_fm, x_test_fm, y_test_fm, class_names_fm)
predictor_fm.predict_random_images(num_predictions=10)

# Bài 2: Xây dựng CNN nhận dạng Cat và Dog

## 1. Thiết lập GPU và Cấu hình

In [ ]:
import tensorflow as tf

class GPUManager:
    """Quản lý và kiểm tra trạng thái GPU, cấu hình tối ưu TensorFlow."""
    def __init__(self):
        pass

    def check_and_configure_gpu(self):
        print("\n--- Kiểm tra và cấu hình GPU ---")
        gpus = tf.config.list_physical_devices('GPU')
        if gpus:
            try:
                print(f"Tìm thấy {len(gpus)} GPU vật lý.")
                for i, gpu in enumerate(gpus):
                    print(f"GPU {i}: {gpu.name}, Kiểu: {tf.config.experimental.get_device_details(gpu)['device_name']}")
                    # Chỉ thiết lập memory growth nếu chưa được thiết lập (hoặc nếu đây là lần đầu tiên)
                    # Đôi khi lỗi xảy ra nếu cố gắng thiết lập lại sau khi đã có hoạt động TF khác
                    try:
                        tf.config.experimental.set_memory_growth(gpu, True)
                    except RuntimeError as e:
                        if "Physical devices cannot be modified after being initialized" in str(e):
                            print(f"Cảnh báo: Memory growth cho GPU {i} đã được thiết lập. {e}")
                        else:
                            raise # Raise other RuntimeErrors

                # Bật Mixed Precision Training
                # Kiểm tra xem policy đã được đặt chưa để tránh cảnh báo
                if tf.keras.mixed_precision.global_policy().name != 'mixed_float16':
                    policy = tf.keras.mixed_precision.Policy('mixed_float16')
                    tf.keras.mixed_precision.set_global_policy(policy)
                    print(f"Đã bật Mixed Precision Training với policy: {policy.name}")
                else:
                    print(f"Mixed Precision Training với policy: {tf.keras.mixed_precision.global_policy().name} đã được bật.")

                # Tối ưu hóa TensorFlow cho GPU
                tf.config.optimizer.set_jit(True) # Enable XLA
                print("Đã bật XLA (Accelerated Linear Algebra).")

                logical_gpus = tf.config.experimental.list_logical_devices('GPU')
                print(f"Tìm thấy {len(gpus)} GPU vật lý, {len(logical_gpus)} GPU logic.")
                print("GPU đã sẵn sàng để huấn luyện với các cấu hình tối ưu.")
                return True
            except RuntimeError as e:
                print(f"Lỗi cấu hình GPU: {e}")
                return False
        else:
            print("Không tìm thấy thiết bị GPU. Đảm bảo bạn đang chạy trên Google Colab GPU và đã kích hoạt Runtime Type 'GPU'.")
            return False

# Khởi tạo và kiểm tra GPU
gpu_manager = GPUManager()
if not gpu_manager.check_and_configure_gpu():
    # Nếu có lỗi nhưng GPU vẫn hoạt động (do đã được cấu hình trước), SystemExit sẽ không được gọi.
    # Nếu thực sự không có GPU hoặc lỗi nghiêm trọng khác, SystemExit vẫn sẽ hoạt động.
    print("Hệ thống không thể cấu hình GPU theo yêu cầu hoặc không tìm thấy GPU. Vui lòng kiểm tra lại môi trường Colab.")
    # Tùy chọn: nếu bạn muốn dừng hoàn toàn nếu GPU không hoạt động, có thể giữ lại SystemExit:
    # raise SystemExit("Không có GPU hoặc cấu hình GPU không thành công. Vui lòng kiểm tra lại môi trường.")



--- Kiểm tra và cấu hình GPU ---
Tìm thấy 1 GPU vật lý.
GPU 0: /physical_device:GPU:0, Kiểu: Tesla T4
Cảnh báo: Memory growth cho GPU 0 đã được thiết lập. Physical devices cannot be modified after being initialized
Đã bật Mixed Precision Training với policy: mixed_float16
Đã bật XLA (Accelerated Linear Algebra).
Tìm thấy 1 GPU vật lý, 1 GPU logic.
GPU đã sẵn sàng để huấn luyện với các cấu hình tối ưu.


In [ ]:
# Khởi tạo và kiểm tra GPU
gpu_manager = GPUManager()
if not gpu_manager.check_and_configure_gpu():
    print("Hệ thống không thể cấu hình GPU theo yêu cầu hoặc không tìm thấy GPU. Vui lòng kiểm tra lại môi trường Colab.")


--- Kiểm tra và cấu hình GPU ---
Tìm thấy 1 GPU vật lý.
GPU 0: /physical_device:GPU:0, Kiểu: Tesla T4
Cảnh báo: Memory growth cho GPU 0 đã được thiết lập. Physical devices cannot be modified after being initialized
Mixed Precision Training với policy: mixed_float16 đã được bật.
Đã bật XLA (Accelerated Linear Algebra).
Tìm thấy 1 GPU vật lý, 1 GPU logic.
GPU đã sẵn sàng để huấn luyện với các cấu hình tối ưu.


## 2. Tải, Tiền xử lý và Chia tách Dữ liệu

In [ ]:
import tensorflow_datasets as tfds
import numpy as np
import math

# Hằng số cấu hình
IMG_SIZE = (150, 150)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

class DatasetLoader:
    """Tải, tiền xử lý và chia tách bộ dữ liệu Cat vs Dog."""
    def __init__(self, img_size=IMG_SIZE):
        self.img_size = img_size
        self.builder = None
        self.info = None
        self.train_ds = None
        self.val_ds = None
        self.test_ds = None
        self.num_train = 0
        self.num_val = 0
        self.num_test = 0

    def _load_dataset(self):
        print("\n--- Đang cố gắng tải bộ dữ liệu 'cats_vs_dogs' từ TensorFlow Datasets ---")
        try:
            self.builder = tfds.builder("cats_vs_dogs")
            self.builder.download_and_prepare()
            self.info = self.builder.info
            print("Đã tải bộ dữ liệu 'cats_vs_dogs' thành công.")
            return True
        except Exception as e:
            print(f"Lỗi khi tải bộ dữ liệu từ tfds: {e}")
            print("Vui lòng kiểm tra kết nối internet hoặc thử lại sau.")
            return False

    def _preprocess_image(self, image, label):
        image = tf.image.resize(image, self.img_size)
        image = tf.cast(image, tf.float32) / 255.0 # Normalize
        return image, label

    def load_and_split_data(self):
        if not self._load_dataset():
            raise SystemExit("Không thể tải bộ dữ liệu Cat vs Dog. Vui lòng kiểm tra lại.")

        # Tổng số mẫu
        total_samples = self.info.splits['train'].num_examples

        # Chia tỷ lệ: Train 70%, Validation 15%, Test 15%
        self.num_train = math.floor(total_samples * 0.7)
        self.num_val = math.floor(total_samples * 0.15)
        self.num_test = total_samples - self.num_train - self.num_val

        print(f"Tổng số mẫu: {total_samples}")
        print(f"Số mẫu huấn luyện (70%): {self.num_train}")
        print(f"Số mẫu kiểm tra (15%): {self.num_test}")
        print(f"Số mẫu kiểm định (15%): {self.num_val}")

        # Tạo các tập dữ liệu
        full_dataset = self.builder.as_dataset(split='train', as_supervised=True)

        self.train_ds = full_dataset.take(self.num_train)
        remaining_ds = full_dataset.skip(self.num_train)
        self.val_ds = remaining_ds.take(self.num_val)
        self.test_ds = remaining_ds.skip(self.num_val)

        # Áp dụng tiền xử lý
        self.train_ds = self.train_ds.map(self._preprocess_image, num_parallel_calls=AUTOTUNE)
        self.val_ds = self.val_ds.map(self._preprocess_image, num_parallel_calls=AUTOTUNE)
        self.test_ds = self.test_ds.map(self._preprocess_image, num_parallel_calls=AUTOTUNE)

        # Đếm số lượng Cat và Dog
        num_cats_train = sum(1 for _, label in self.train_ds if label == 0)
        num_dogs_train = sum(1 for _, label in self.train_ds if label == 1)
        print(f"\nSố lượng ảnh Cat trong tập huấn luyện: {num_cats_train}")
        print(f"Số lượng ảnh Dog trong tập huấn luyện: {num_dogs_train}")

        num_cats_test = sum(1 for _, label in self.test_ds if label == 0)
        num_dogs_test = sum(1 for _, label in self.test_ds if label == 1)
        print(f"Số lượng ảnh Cat trong tập kiểm tra: {num_cats_test}")
        print(f"Số lượng ảnh Dog trong tập kiểm tra: {num_dogs_test}")

        num_cats_val = sum(1 for _, label in self.val_ds if label == 0)
        num_dogs_val = sum(1 for _, label in self.val_ds if label == 1)
        print(f"Số lượng ảnh Cat trong tập kiểm định: {num_cats_val}")
        print(f"Số lượng ảnh Dog trong tập kiểm định: {num_dogs_val}")

        # Cấu hình hiệu suất cho các dataset
        self.train_ds = self.train_ds.cache().shuffle(1000).batch(BATCH_SIZE).prefetch(buffer_size=AUTOTUNE)
        self.val_ds = self.val_ds.cache().batch(BATCH_SIZE).prefetch(buffer_size=AUTOTUNE)
        self.test_ds = self.test_ds.cache().batch(BATCH_SIZE).prefetch(buffer_size=AUTOTUNE)

        print("Dữ liệu đã được tải, tiền xử lý và chia tách thành công.")
        return self.train_ds, self.val_ds, self.test_ds, self.info

# Khởi tạo và tải dữ liệu
dataset_loader = DatasetLoader()
train_ds, val_ds, test_ds, info = dataset_loader.load_and_split_data()


## 3. Trực quan hóa dữ liệu

In [ ]:
import matplotlib.pyplot as plt

class DataVisualizer:
    """Trực quan hóa dữ liệu Cat vs Dog."""
    def __init__(self, train_ds, info):
        self.train_ds = train_ds
        self.info = info
        self.class_names = ['cat', 'dog'] # Mặc định cho cats_vs_dogs

    def display_sample_images(self, num_images=10):
        print(f"\n--- Hiển thị {num_images} ảnh mẫu từ tập huấn luyện ---")
        plt.figure(figsize=(15, 8))
        for i, (image, label) in enumerate(self.train_ds.unbatch().take(num_images)):
            ax = plt.subplot(2, 5, i + 1)
            plt.imshow(image.numpy())
            plt.title(self.class_names[label.numpy()])
            plt.axis("off")
            if i == num_images - 1: # Break after taking num_images
                break
        plt.suptitle(f'{num_images} ảnh mẫu từ bộ dữ liệu Cat vs Dog', y=1.02)
        plt.show()

# Khởi tạo và hiển thị ảnh mẫu
data_visualizer = DataVisualizer(train_ds, info)
data_visualizer.display_sample_images(num_images=10)


## 4. Tăng cường dữ liệu (Data Augmentation)

In [ ]:
class DataAugmentation:
    """Thực hiện Data Augmentation cho dữ liệu huấn luyện."""
    def __init__(self, img_size=IMG_SIZE):
        self.img_size = img_size
        self.data_augmentation_layer = self._build_augmentation_layer()
        print("Đã cấu hình Data Augmentation.")

    def _build_augmentation_layer(self):
        return tf.keras.Sequential([
            tf.keras.layers.RandomRotation(factor=0.15),
            tf.keras.layers.RandomZoom(height_factor=0.1, width_factor=0.1),
            tf.keras.layers.RandomFlip("horizontal"),
            tf.keras.layers.RandomContrast(factor=0.1)
        ], name="data_augmentation")

    def apply_augmentation(self, dataset):
        """Áp dụng data augmentation cho một dataset."""
        return dataset.map(lambda x, y: (self.data_augmentation_layer(x, training=True), y), num_parallel_calls=AUTOTUNE)

# Khởi tạo DataAugmentation và áp dụng cho tập huấn luyện
data_augmentor = DataAugmentation(img_size=IMG_SIZE)
train_ds_augmented = data_augmentor.apply_augmentation(train_ds)

# Ghi đè train_ds ban đầu bằng train_ds_augmented để sử dụng trong huấn luyện
train_ds = train_ds_augmented

print("Đã áp dụng Data Augmentation cho tập huấn luyện.")


## 5. Xây dựng mô hình CNN

In [ ]:
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Dropout, Dense, GlobalAveragePooling2D, Input
from tensorflow.keras.optimizers import Adam

class CNNModel:
    """Xây dựng và biên dịch mô hình CNN cho phân loại Cat vs Dog."""
    def __init__(self, input_shape, num_classes=2):
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.model = self._build_model()

    def _build_model(self):
        inputs = Input(shape=self.input_shape)

        # Conv Block 1
        x = Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same')(inputs)
        x = BatchNormalization()(x)
        x = MaxPooling2D((2, 2))(x)
        x = Dropout(0.2)(x)

        # Conv Block 2
        x = Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same')(x)
        x = BatchNormalization()(x)
        x = MaxPooling2D((2, 2))(x)
        x = Dropout(0.3)(x)

        # Conv Block 3
        x = Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_uniform', padding='same')(x)
        x = BatchNormalization()(x)
        x = MaxPooling2D((2, 2))(x)
        x = Dropout(0.4)(x)

        # Global Average Pooling
        x = GlobalAveragePooling2D()(x)

        # Dense layers
        x = Dense(256, activation='relu', kernel_initializer='he_uniform')(x)
        x = BatchNormalization()(x)
        x = Dropout(0.5)(x)

        # Output layer
        outputs = Dense(self.num_classes, activation='softmax', dtype='float32')(x) # Output layer should be float32 for mixed precision

        model = Model(inputs=inputs, outputs=outputs)

        optimizer = Adam(learning_rate=0.001)
        model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        print("Mô hình CNN đã được xây dựng và biên dịch.")
        model.summary()
        return model

    def get_model(self):
        """Trả về mô hình CNN đã được xây dựng."""
        return self.model

# Lấy một batch từ train_ds để suy ra input_shape
for image_batch, label_batch in train_ds.take(1):
    input_shape_from_data = image_batch.shape[1:]
    break

cnn_model_builder = CNNModel(input_shape=input_shape_from_data, num_classes=info.features['label'].num_classes)
model = cnn_model_builder.get_model()


## 6. Huấn luyện mô hình

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

class ModelTrainer:
    """Huấn luyện mô hình CNN với các callbacks."""
    def __init__(self, model, train_ds, val_ds, model_name='cat_dog_cnn_model.keras'):
        self.model = model
        self.train_ds = train_ds
        self.val_ds = val_ds
        self.model_name = model_name
        self.history = None
        self.callbacks = self._get_callbacks()

    def _get_callbacks(self):
        early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
        model_checkpoint = ModelCheckpoint(filepath=self.model_name, monitor='val_accuracy', save_best_only=True, verbose=1)
        reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.00001, verbose=1)
        return [early_stopping, model_checkpoint, reduce_lr]

    def train(self, epochs=15):
        print("Bắt đầu huấn luyện mô hình...")
        self.history = self.model.fit(
            self.train_ds,
            epochs=epochs,
            validation_data=self.val_ds,
            callbacks=self.callbacks,
            verbose=1
        )
        print("Huấn luyện hoàn tất.")
        return self.history

    def plot_history(self):
        """Vẽ biểu đồ Accuracy và Loss của quá trình huấn luyện."""
        if self.history is None:
            print("Chưa có lịch sử huấn luyện để vẽ biểu đồ.")
            return

        plt.figure(figsize=(12, 4))
        plt.subplot(1, 2, 1)
        plt.plot(self.history.history['accuracy'], label='Training Accuracy')
        plt.plot(self.history.history['val_accuracy'], label='Validation Accuracy')
        plt.title('Training and Validation Accuracy')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.grid(True)

        plt.subplot(1, 2, 2)
        plt.plot(self.history.history['loss'], label='Training Loss')
        plt.plot(self.history.history['val_loss'], label='Validation Loss')
        plt.title('Training and Validation Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True)

        plt.tight_layout()
        plt.show()

# Khởi tạo ModelTrainer và huấn luyện mô hình
model_trainer = ModelTrainer(
    model=model,
    train_ds=train_ds,
    val_ds=val_ds
)
history = model_trainer.train(epochs=50)
model_trainer.plot_history()


## 7. Đánh giá mô hình

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import numpy as np

class ModelEvaluator:
    """Đánh giá mô hình đã huấn luyện và hiển thị kết quả."""
    def __init__(self, model, test_ds, class_names):
        self.model = model
        self.test_ds = test_ds
        self.class_names = class_names

    def evaluate(self):
        print("\nĐang đánh giá mô hình trên tập kiểm tra...")
        loss, accuracy = self.model.evaluate(self.test_ds, verbose=0)
        print(f"Độ chính xác trên tập kiểm tra: {accuracy:.4f}")
        print(f"Loss trên tập kiểm tra: {loss:.4f}")
        return loss, accuracy

    def display_classification_report(self):
        print("\nHiển thị Báo cáo phân loại:")
        # Lấy nhãn thực tế và nhãn dự đoán
        y_true_labels = []
        y_pred_labels = []
        for images, labels in self.test_ds:
            predictions = self.model.predict(images, verbose=0)
            y_pred_labels.extend(np.argmax(predictions, axis=1))
            y_true_labels.extend(labels.numpy())

        print(classification_report(y_true_labels, y_pred_labels, target_names=self.class_names))

    def display_confusion_matrix(self):
        print("\nHiển thị Ma trận nhầm lẫn:")
        y_true_labels = []
        y_pred_labels = []
        for images, labels in self.test_ds:
            predictions = self.model.predict(images, verbose=0)
            y_pred_labels.extend(np.argmax(predictions, axis=1))
            y_true_labels.extend(labels.numpy())

        cm = confusion_matrix(y_true_labels, y_pred_labels)

        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='g', cmap='Blues', xticklabels=self.class_names, yticklabels=self.class_names)
        plt.xlabel('Dự đoán')
        plt.ylabel('Thực tế')
        plt.title('Ma trận nhầm lẫn')
        plt.show()

# Khởi tạo ModelEvaluator và đánh giá mô hình
# Lấy tên lớp từ info.features['label'].names
class_names_cat_dog = info.features['label'].names
model_evaluator = ModelEvaluator(model, test_ds, class_names_cat_dog)
model_evaluator.evaluate()
model_evaluator.display_classification_report()
model_evaluator.display_confusion_matrix()


## 8. Dự đoán ảnh chó/mèo

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import matplotlib.image as mpimg
import io
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import requests # Để tải ảnh từ URL

class Predictor:
    """Thực hiện dự đoán trên ảnh mới được tải lên, từ URL hoặc từ camera."""
    def __init__(self, model, img_size, class_names):
        self.model = model
        self.img_size = img_size
        self.class_names = class_names

    def _preprocess_image(self, img_array):
        # Đảm bảo ảnh có 3 kênh màu nếu cần (grayscale image will be converted to 3 channels)
        if img_array.shape[-1] == 1: # If grayscale, convert to 3 channels
            img_array = tf.image.grayscale_to_rgb(img_array)

        img_array = tf.image.resize(img_array, self.img_size)
        img_array = tf.expand_dims(img_array, 0) # Create a batch
        img_array = tf.cast(img_array, tf.float32) / 255.0 # Normalize and cast
        return img_array

    def _display_prediction(self, processed_img, predictions):
        predicted_class_index = np.argmax(predictions)
        predicted_label = self.class_names[predicted_class_index]
        confidence = np.max(predictions) * 100

        plt.figure(figsize=(4, 4))
        # Ensure the image is in the correct format for display
        display_img = processed_img[0].numpy() if tf.is_tensor(processed_img[0]) else processed_img[0]
        plt.imshow(display_img)
        plt.title(f"Dự đoán: {predicted_label} ({confidence:.2f}%) ", fontsize=10)
        plt.axis('off')
        plt.show()

        print(f"Kết quả dự đoán: {predicted_label} với độ tin cậy {confidence:.2f}%")

    def predict_from_local_file(self, image_path):
        print(f"\n--- Dự đoán ảnh từ file cục bộ: {image_path} ---")
        img = tf.keras.preprocessing.image.load_img(image_path)
        img_array = tf.keras.preprocessing.image.img_to_array(img)
        processed_img = self._preprocess_image(img_array)

        predictions = self.model.predict(processed_img)
        self._display_prediction(processed_img, predictions)

    def predict_from_url(self, image_url):
        print(f"\n--- Dự đoán ảnh từ URL: {image_url} ---")
        try:
            response = requests.get(image_url)
            response.raise_for_status() # Kiểm tra lỗi HTTP
            img = tf.keras.preprocessing.image.load_img(io.BytesIO(response.content))
            img_array = tf.keras.preprocessing.image.img_to_array(img)
            processed_img = self._preprocess_image(img_array)

            predictions = self.model.predict(processed_img)
            self._display_prediction(processed_img, predictions)
        except requests.exceptions.RequestException as e:
            print(f"Lỗi khi tải ảnh từ URL: {e}")
        except Exception as e:
            print(f"Lỗi khi xử lý ảnh từ URL: {e}")

    def _take_photo(self, filename='photo.jpg', quality=0.8):
        js = Javascript('''
            async function takePhoto(quality) {
                const div = document.createElement('div');
                const capture = document.createElement('button');
                capture.textContent = 'Capture Photo';
                div.appendChild(capture);

                const video = document.createElement('video');
                video.style.display = 'block';
                const stream = await navigator.mediaDevices.getUserMedia({video: true});

                document.body.appendChild(div);
                div.appendChild(video);
                video.srcObject = stream;
                await video.play();

                // Resize the output to fit the display
                google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

                // Wait for Capture to be clicked.
                await new Promise((resolve) => capture.onclick = resolve);

                const canvas = document.createElement('canvas');
                canvas.width = video.videoWidth;
                canvas.height = video.videoHeight;
                canvas.getContext('2d').drawImage(video, 0, 0);
                stream.getVideoTracks()[0].stop();
                div.remove();
                return canvas.toDataURL('image/jpeg', quality);
            }
            ''')
        display(js)
        data = eval_js('takePhoto({})'.format(quality))
        binary = b64decode(data.split(',')[1])
        with open(filename, 'wb') as f:
            f.write(binary)
        return filename

    def predict_from_camera(self):
        try:
            filename = self._take_photo()
            print(f"Ảnh chụp đã lưu tại: {filename}")

            img = tf.keras.preprocessing.image.load_img(filename)
            img_array = tf.keras.preprocessing.image.img_to_array(img)
            processed_img = self._preprocess_image(img_array)

            predictions = self.model.predict(processed_img)
            self._display_prediction(processed_img, predictions)
        except Exception as err:
            print(str(err))

# Khởi tạo Predictor
predictor = Predictor(model, IMG_SIZE, class_names_cat_dog)


### 8. Dự đoán với ảnh tải lên từ máy tính

In [ ]:
from google.colab import files

print("Vui lòng tải lên ảnh của bạn (Chó hoặc Mèo):")
uploaded = files.upload()

for filename in uploaded.keys():
    predictor.predict_from_local_file(filename)
